# **Introduction (Assignment 1, Deadline 23/10/2026 at 08:00 AM)**

Students write text here, explaining:
 - What it means to preprocess data and what is a preprocessing pipeline
 - What are the main artefacts that can be found in the TMS-EEG signal and their causes
 - **Optional**: a personal comment or opinion on the problems posed by preprocessing, based on what you heard in class
   
Data preprocessing is a set of operations aimed at separating the signal, that is the fraction of scalp voltage generated by the cortex in response to the TMS pulse, from the noise, that is a signal generated by other agents such as muscles and electronic instrumentation and that is not interesting in respect to the analysis. A preprocessing pipeline consists of a series of multiple transformations done in a precise sequence in order to separate signal from noise. Nowadays there is not a standard preprocessing pipeline; however, a broad and schematic classification can be made based upon the philosophical approach underlying the preprocessing pipeline. Conservationist approaches try to preserve as much information as possible, taking the risk of having more noisy data; instead, interventionist approaches act heavily on the signal, prioritizing the minimization of noise at the risk of losing some significant information. The main artefacts that can be found in the raw TMS-EEG signal are eyeblinks artefacts, muscles artefacts and TMS pulse artefacts. Eyes artefacts are caused by ocular movements, they are much larger than the rest of the signal and have a limited temporal duration, they are therefore relatively easily localized and are more visible in the signal registered in the frontal channels. Muscle artefacts have high amplitude, high frequency and are present in several channels; they generate a signal that is noticeably larger than the cortical one and cancels the latter. A third kind of artefact is the one generated by the TMS pulse at the time of the stimulation. It is a very strong signal that completely overrides the cortical signal and has to be eliminated in order to make the data usable; this part of the preprocessing pipeline is done using interpolation.
The subjectivity in the choice of the epistemological approach to the preprocessing pipeline, that can be more conservationist or more interventionist, might be a problem in terms of the reproducibility of scientific findings. I find particularly interesting how the physiological needs determine the methodological approach and impose certain trade-offs, however without a shared standardization the prospective of a shared knowledge might be more difficult. For what it concerns reproducibility, another issue is the need of transparency in every step of the pipeline when sharing a scientific research.


In [2]:
from pathlib import Path                            # to build a bridge between Python and the filesystem 

import numpy as np                                  # to perform array operations
import matplotlib                                   # this and the following to draw plots
import matplotlib.pyplot as plt
matplotlib.use('Qt5Agg')                            # to make plots interactive

import mne                                          # to read and manipulate EEG data

from scripts import utils                           # custom functions to shorten the code in this notebook

# **1. Basic Preprocessing (Assignment 1, Deadline 23/10/2026 at 08:00 AM)**

Students write text here, listing the operations that we categorise as "basic preprocessing".

If you cannot remember what the steps are, look at the code in the cells below and you will recall.

Basic preprocessing can be intended as a small set of essential operations that can be summarized as follows:
 - Loading the data and making them available to the software
 - Inspecting data
 - Adjusting data
 - Interpolating
 - High-pass filtering
 

## **1.1. Data loading and availability**

**Optional but welcome:** students write here any considerations about this step.

In the first part of this step there is the definition of two variables: data_dir is the name chosen for the directory where the data are stored and subject_and_condition is a defined variable that in this case identifies the subject 02 in condition 1.
In the string with the definition of file_of_interest we can see the suffix .vhdr. In the BrainVision format, that is the one used in this study, information is spread across 3 files:
 1) .eeg = eeg time series
 2) .vhdr = metadata, they give context to the data
 3) .vmrk = information about event marker

In the second block of code we can see MNE, an open-source packet that is the gold standard for EEG analysis.
In this step data are imported and made available to the software for subsequent operations.


In [4]:
data_dir = Path("data")
subject_and_condition = "S02C1_M1"

file_of_interest = list(data_dir.glob(pattern=f"{subject_and_condition}.vhdr"))
print(file_of_interest, "|", type(file_of_interest), "|", file_of_interest[0])

[WindowsPath('data/S02C1_M1.vhdr')] | <class 'list'> | data\S02C1_M1.vhdr


In [5]:
eeg_data = mne.io.read_raw_brainvision(vhdr_fname=file_of_interest[0])

Extracting parameters from data\S02C1_M1.vhdr...
Setting channel info structure...


## **1.2. Channels dropping (adjusting data)**

**Compulsory:** students write here the functional significance of this step (that is, why we do it).

This step consists in the elimination of unwanted channels. The discrimination between wanted and unwanted channels is done leveraging their denomination. Central channels end with z (zenit), channels on the right with even numbers and those on the left with odd numbers. The numbers are assigned in an ascending order from medial to lateral regions. The subsequent block of code defines as channels to be dropped those whose last character of the name is not a number and is not a "z". Here the output states that the channels EOG (ElectroOculoGram - ocular activity) and FDI (First Dorsal Interosseous muscle) must be dropped, that is because they are unwanted channels they are not useful to the data analysis since they do not register signal of interest. 

In [6]:
channels_to_drop = []

for channel_name in eeg_data.ch_names:
    try:
        int(channel_name[-1])
    except ValueError:
        if channel_name[-1] != "z": 
            channels_to_drop.append(channel_name)
print(f"Channels to drop: {channels_to_drop}")

eeg_data = eeg_data.drop_channels(ch_names=channels_to_drop)

Channels to drop: ['EOG', 'FDI']


## **1.3. Data inspection**

**Compulsory:** students write here the functional significance of this step (that is, why we do it).

Inspecting the data provides information such as their shape, content and aquisition parameters. The grid below shows information divided in four sections: general (contains metadata), acquisition, channels and filters. The "acquisition" and "channels" section are particularly relevant for data inspection since they contain information about respectively the temporal and spatial dimension of the data. In the "acquisition" section are indicated the duration of the recording (hh: mm :ss), the sampling frequency, that is how many samples per second are saved by the software  (measured in Hz), and the time points. Regarding the latter, EEG data are structurally organized in a matrix in which each element is defined in a time point measured in array units. It is worth noting that all the acquisition parameters have units of measurment that have the concept of time in common, thus making possible the application of formulas to convert between them, mapping between the "real world time" and the data structure.
In order to investigate the spatial dimension, data are standardized using a built-in MNE's easycap-M1 montage, that is compatible with the physical cap used in the study from which the present data are taken. Doing so makes possible the analysis at the level of the channels which is essential for source reconstruction and topoplots.

In [7]:
easycap_m1_montage = mne.channels.make_standard_montage(kind="easycap-M1")
easycap_m1_montage.plot()
eeg_data.set_montage(montage=easycap_m1_montage,
                     on_missing="ignore")

<RawBrainVision | S02C1_M1.eeg, 70 x 3307600 (661.5 s), ~85 KiB, data not loaded>

**Compulsory:** here, students describe and comment the plot generated by the code below. In particular, I would like to read:

- What each row represents
- What are the $x$ and $y$ of each row, and their units of measurement
- A description of the artefacts you see, and their likely causes

The code below generates a plot that is a visual rapresentation of the data matrix. Each row of the plot represents a channel. On the x axis there is time, measured in seconds (s). The y axis of each row is the signal intensity, measured in microvolts (μV); the amplitude visualization span is 40μV peak-to-peak.
The main artefacts visible on the plot are:
1) muscle artefacts, signals with high amplitude, high frequency and ubiquitous; caused by muscular activity
2) eyeblinks artefacts, relatively smaller than muscle artefacts but still larger than cortical signal, more present in the frontal area; caused by ocular activity
3) TMS pulse artefact, a high frequency signal localized at the time of stimuli presentations; caused by the electric current induced by the electromagnetic pulse.


In [8]:
eeg_data.plot()

Using matplotlib as 2D backend.
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`


<MNEBrowseFigure size 1920x986 with 4 Axes>

Channels marked as bad:
none


**Compulsory:** here, students define event markers and explain why they are important for EEG data analysis
**Optional:** any comments that go beyond a mere definition are welcome. For example, you could make comments about the structure of event arrays and what it tells us about the nature of events

Event markers are contained in the .vmrk file and provide information about the ordinal number, type, name, time of occurrence and duration of a stimulus, and also information about the channels involved. Time of occurrence and duration of a stimulus are not measured in seconds but in data points and allow the precise localization of the stimulus in the EEG data matrix. These markers are important for the TMS-EEG signal analysis since they provide the localization of the TMS pulse in the array and therefore of the TMS pulse artefact. This artefact is in fact so strong that it overrides the relevant signal and it needs to be eliminated from the data in order to do the analysis. The following block of code uses MNE to extract marker information from the eeg data structures and translates them into a Python-friendly format. 





In [9]:
events_from_annotations, events_dict = mne.events_from_annotations(raw=eeg_data)
events_from_annotations = events_from_annotations[2:]

Used Annotations descriptions: ['New Segment/', 'Stimulus/S 54', 'Stimulus/S255']


## **1.4. Interpolation**

**Compulsory:** students write here the functional significance of this step (that is, why we do it).

Interpolation is a mathematical technique that estimates the missing values of a function based on the preceding and following values of such function. As mentioned above, the TMS pulse artefacts heavily affect the data and they need to be removed. Doing so will result in loss of the cortical signal at the time of the pulse. Interpolation allows for the estimation of this lost information. Before interpolation, an essential step is segmenting the data, that is dividing the EEG trace into epochs centered on the event (the TMS pulse). 

In [10]:
pre_interpolation_epochs = mne.Epochs(raw=eeg_data,
                                      events=events_from_annotations,
                                      tmin=-1.1,
                                      tmax=0.5,
                                      baseline=None)
pre_interpolation_epochs_tep = pre_interpolation_epochs.average()
pre_interpolation_epochs_tep.plot();

Not setting metadata
200 matching events found
No baseline correction applied
0 projection items activated


In [11]:
post_interpolation_eeg = mne.io.read_raw(fname="data/post_2_5_interpolation_eeg.fif",
                                         preload=True)
post_interpolation_epochs = mne.Epochs(raw=post_interpolation_eeg,
                                       events=events_from_annotations,
                                       tmin=-1.1,
                                       tmax=0.5,
                                       baseline=None)
post_interpolation_tep = post_interpolation_epochs.average()
post_interpolation_tep.plot();

Opening raw data file data/post_2_5_interpolation_eeg.fif...
    Range : 0 ... 3307599 =      0.000 ...   661.520 secs
Ready.
Reading 0 ... 3307599  =      0.000 ...   661.520 secs...
Not setting metadata
200 matching events found
No baseline correction applied
0 projection items activated


## **1.5. High-pass filtering**

**Compulsory:** students write here the functional significance of this step 

EEG data consist of various oscillations at different frequencies. Depending on the goal of the study, some frequencies might not be wanted and there is the need to eliminate them. Filters are methodological tools that are used to study waves at specific frequencies of interest or above or below certain frequencies. Some of the main filters used in data analysis are:
1) high-pass filter, that filters out frequencies below a certain value
2) low-pass filter, that filters out frequencies above a certain value
3) band-pass , that filters out frequencies outside a certain range
4) notch filter, a band stop filter that weakens frequencies inside a certain range (used at 50Hz in Europe and at 60Hz in the United States to eliminate the artifacts generated by the electrical current)
The following block of code applies a high-pass filter at 0.1 Hz to the data, weakening the frequencies lower than 0.1Hz.

Filters work based on the mathematical operation of convolution.





In [12]:
post_filtering_eeg = mne.filter.filter_data(data=post_interpolation_eeg.get_data(),
                                            sfreq=post_interpolation_eeg.info["sfreq"],
                                            l_freq=0.1,
                                            h_freq=None,
                                            method="iir",
                                            iir_params=None,
                                            copy=True,
                                            phase="zero")
post_filtering_eeg = mne.io.RawArray(data=post_filtering_eeg,
                                     info=post_interpolation_eeg.info)
post_filtering_epochs = mne.Epochs(raw=post_filtering_eeg,
                                   events=events_from_annotations,
                                   tmin=-1.1,
                                   tmax=0.5,
                                   baseline=None)
post_filtering_tep = post_filtering_epochs.average()
post_filtering_tep.plot();

Setting up high-pass filter at 0.1 Hz

IIR filter parameters
---------------------
Butterworth highpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 8 (effective, after forward-backward)
- Cutoff at 0.10 Hz: -6.02 dB

Creating RawArray with float64 data, n_channels=70, n_times=3307600
    Range : 0 ... 3307599 =      0.000 ...   661.520 secs
Ready.
Not setting metadata
200 matching events found
No baseline correction applied
0 projection items activated


# **2. Manual Artifact Rejection & Bad Channels Rejection** 
# **(Assignment 2, Deadline 30/10/2026 at 08:00 AM)**

## **2.1. Rationale**

**Compulsory:** students write here the functional significance of this step (that is, why we do it) and expand on the criteria to follow for its execution

This step contains the two processes of Manual Artifact Rejection (MAR) and Bad Channels Rejection (BCR).
These two methods are the simplest form of data cleaning and consist in the visual inspection of the EEG data set and in the removal of compromised epochs or bad channels. Since the statistical average is heavily affected by outliers, it is essential to remove extreme values (present in bad trials and bad channels) in order to preserve the reliability of the statistical average, e.g. when computing the average of MEPs.

The cases in which MAR and BCR are used are the following:

 1- To remove the largest artifacts or bad channels *before ICA*
 
 2- To remove leftovers artifacts or bad channels *after automatic procedures*
 
 3- To remove the *most obvious* artifacts or bad channels
 
Even though they are used in the same scenarios, the two procedures have different use criteria.


Regarding **MAR**, the aim is to inspect the data and flag epochs for rejection.
The cases in which MAR should be used are the following:

 1- In case of high frequency, high amplitude artifacts that cannot be of cerebral origin
 
 2- The artifacts should be circumscribed in time
 
 3- The artifacts should affect multiple channels
 

Passing now to **BCR**, this procedure removes from the analysis channels that are considered as bad according to the following criteria:

 1- They report high frequency, high amplitude artifacts that cannot be of cerebral origin
 
 2- The artifacts should not be circumscribed in time
 
 3- The artifacts affect a single channel for a relatively long lasting period
 
These artifacts can be found in the case of poor contact between the electrode and the scalp or in the case of cable damage.
Rejected channels can be interpolated with their neighbours channels due to the relatively low spatial resolution of the EEG signal. However, it is worth noting that it is not recommended to yield scientific conclusions from interpolated channels and in the eventuality of consistent line noise in channels that are crucial for the experiment it might be advisable to exclude the subject from the experiment.

## **2.2. Execution**

**Compulsory:** students reject bad epochs and bad channels following appropriate criteria. Provide a list of what you reject and why (for example: _"I choose to reject channel $x$ because it is affected by line noise and epoch $y$ because it is affected by a large mechanical artifact"_)

I choose to reject the following channels due to line noise: F7, FC3, C3, C1, TP8, P10, PO10, O1, Oz.

I choose to reject the epochs 4, 8, 9 , 49 because they are affected by large mechanical artifacts.


In [17]:
post_filtering_epochs = mne.read_epochs(fname="data/post_filtering_epo.fif", 
                                        preload=True)
post_filtering_epochs.plot();

Reading C:\Users\Giulia\Desktop\brainstim-multimodal\data\post_filtering_epo.fif ...
    Found the data of interest:
        t =   -1100.00 ...     500.00 ms
        0 CTF compensation matrices available
Not setting metadata
200 matching events found
No baseline correction applied
0 projection items activated
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
Dropped 0 epochs: 
The following epochs were marked as bad and are dropped:
[]
Channels marked as bad:
none


In [16]:
post_filtering_epochs

<EpochsFIF | 200 events (all good), -1.1 – 0.5 s (baseline off), ~854.7 MiB, data loaded,
 '54': 200>

# **3. Independent Component Analysis (ICA)**

# **(Assignment 3, Deadline 06/11/2026 at 08:00 AM)**

## **3.1. Rationale**

**Compulsory:** students write here the functional significance of this step (that is, why we do it) and expand on its rationale, focusing on discussing the independence assumption and its application to brain data.

The EEG signal is composed by all the sources that might generate electrical activity. As previously seen, the recorded signal contains a part of noise, that is non-cerebral activity. ICA is a mathematical framework that permits to factor the signal in separate components in order to identify the non-cerebral signal sources and eliminate the part of the signal attributed to those components. The three **constrainsts** imposed to the factors, or components, in order to apply ICA are the following:

1. they must be statistically independent (i.e., independence assumption)
2. the original signal can be expressed as a linear combination of said factors
3. their probability densities are non-Gaussian

In practice, ICA can be summarized in the following **steps**:
1. fitting of the ICA model to the data to factorize the signal in the components
2. visualization of the resulting components using their topoplots and voltage time-series (and eventually complementary plots like the components' power spectra)
3. evaluations of the topoplots to identify sources of non-brain activity
4. exclusion of these non-brain components
5. reconstruction of the signal by linearly combining the remaining components (see constraint number 2)

The application of the ICA model to the brain signals stems from the assumption that the activities of the different sources that compose the observed signal are independent of each other (see constraint number 1).

So, if we express the signal as a linear comination (that is the weighted sum) of the **electrical activity of one area + electrical activity of eye movements + electrical activity of heartbeat + line noise + ...**(add anything that could generate electrical activity visible in the EEG signal), if we apply an ICA algorithm to this signal we are assuming that these activities are completely independent.

Independence between two variables can be defined as when information about one of them gives no information about the other and vice-versa; from a probability density function perspective, the joint probability density of two independent variables equals to the product of their marginal densities. 

$p(s_{1},s_{2})=p_{1}(s_{1})p_{2}(s_{2})$ .

This can be extended to any *n* number of variables. 

Physiologically, these activities (excluding line noise) are not actually independent since the brain is an organ of the body, so we can consider it as a node (or nodes) embedded in the body network, and for example there are evidences that show that MEPs vary as a function of the cardiac cycle (Syrov et al., 2026). The independence assumption is then not really correct from a statistical and theoretical-informational perspective and there is the need to distinguish the mathematical framework from the physiological reality.

In the theoretical framework of ICA, the main goal is to find a transformation that allows to derive the original signal sources starting from the observed signal (the so called cocktail-party problem). The observed signal can be thought of as a mix of the signals from the original sources. The transformation to arrive from the signal generated by the original sources to the observed signal can be thought of as a process that mixes the signal from the independent souces. That is called the mixing matrix. To derive the independent sources we want to unmix the signal and that is done using the inverse of the mixing matrix, also called unmixing matrix. So if $X$ is the EEG data matrix we can multiply it by the unmixing matrix $W$ to obtain the component matrix $S$ ($S=WX$). And multiplyng $S$ by $W^{-1}$ we obtain the mixed signal $X$ ($X=W^{-1}S$). Note that the rows and columns of $X$ and $S$ are respectively the channels and time points and the components and time points. The components matrix obtained with the unmixing is a solution that maximises the independence between the components. In particular, multiplying $X$ by $W$ ($S=WX$) maximizes the non-Gaussianity of the probability distributions of the rows of $S$, and considering the Central Limit Theory non-Gaussianity can be considered a measure of independence (see Hyva¨rinen & Oja, 2000). So ICA maximises some measures of non-Gaussianity like kurtosis or negentropy, or minimizes the mutual information between the components (that is equivalent to maximizing negentropy).

ICA can be functionally and empirically useful for identifying and extracting noise sources, however it finds components that *do* have mathematical meaning but at the same time *do not* have neuroscientific meaning, so an accurate and critical examination of the components topoplots and time series is crucial in order to make a distinction between these two levels of analysis.

Notwithstanding this limitation of ICA applied to brain signals, eyeblinks are a kind of artifact on which this method is often used. That is because, even though they are part of the body and originate from the brain, usually eyeblinks do not have a meaningful connection with the dynamics that are interest of study of most EEG experiments. Also, 1) they have a relative large amplitude and they have to be removed since the statistical average (used for example when calculating MEPs) is sensitive to outliers and 2) there is a lack of better methods, since automated alternatives like deconvolution are uncommon, and MAR would be too aggressive and cause significative information loss.

Applicationally, ICA can be considered an optimization problem. The goal is to find the optimum unmixing matrix among all the infinite number of matrix that could work for the unmixing. This is not a deterministic process and implies a certain amount of *randomness*. In fact, depending on the initial unmixing matrix the algorithm could reach different local maxima of non-Gaussianity. (Furthermore, it is worth noting that due to the nature of the $S=WX$ equation, there is ambiguity about the sign and the order of the components.)

This optimization problem is approached using an iterative process that unfolds in the following steps:

1. performing of the unmixing with some matrix composed of pseudorandom values
2. measurement of the dependence of the resulting components
3. adjustment of the unmixing and try of another matrix that will increase the independence between the components
4. all the three previous steps are repeated until 1)dependence is at a (local?) minimum or 2)a predetermined number of iterations is reached.

Regarding the first step, the starting unmixing matrix will vary between different runs and different computers. In order to increase the likelihood of obtaining reproducible results, it is important to set a random seed. This seed will be object of complex computations and the obtained results are considered pseudorandom, and they will be the values of the first unmixing matrix. Different random seeds will be the starting points that arrive to different optima unmixing matrix so being transparent and sharing the random seed is crucial to the clarity and reproducibility of the scientific results. Furthermore, setting the process to end when a good solution is reached instead of when a certain number of iterations is reached is also important to the sake of reproducibility.


## **3.2. Components Selection**

**Compulsory:** select one or more components whose most likely source are eye movements and justify their choices, reporting the number(s) of the selected component(s).
When justifying your choice, discuss both the scalp topography and the time course of the component(s).
Selecting no components is a legitimate choice but it must be justified in writing. 

I chose to select the component ICA000 because I suppose that it represents an eyeblink artifact. The topoplot shows a unipolar activation in the frontal part of the sagittal midline and the corresponding voltage-time series shows relatively large and short-lasting peaks with a quasi-random distribution. The component power spectrum shows values that are higher at lower frequencies and then decrease.

I chose to select the component ICA026 because I suppose it is an horizontal eye-movement. The topoplot shows a simmetric bipolar concentration of activity in the frontal area above the eyes. The voltage-time series shows short-lasting peaks of activation alternating in both directions.

In [2]:
post_rejection_epochs = mne.read_epochs(fname="data/post_rejection_epo.fif", 
                                        preload=True)
ica = mne.preprocessing.read_ica(fname="data/fitted_ica.fif")

Reading C:\Users\Giulia\Desktop\brainstim-multimodal\data\post_rejection_epo.fif ...
    Found the data of interest:
        t =   -1100.00 ...     500.00 ms
        0 CTF compensation matrices available
Not setting metadata
188 matching events found
No baseline correction applied
0 projection items activated
Reading C:\Users\Giulia\Desktop\brainstim-multimodal\data\fitted_ica.fif ...
Now restoring ICA solution ...
Ready.


In [11]:
ica.plot_components(picks=range(30));

In [4]:
ica.plot_sources(inst=post_rejection_epochs, picks=range(30));

Not setting metadata
188 matching events found
No baseline correction applied
0 projection items activated
Using matplotlib as 2D backend.
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`


In [9]:
ica.exclude = [0, 26] # insert a list of the components to reject. For example, insert [21] if you want to reject component 21 (just a random example)
ica.apply(post_rejection_epochs.load_data())
post_ica_epochs = post_rejection_epochs
post_ica_tep = post_ica_epochs.average()
post_ica_tep.plot();

Applying ICA to Epochs instance
    Transforming to ICA space (64 components)
    Zeroing out 2 ICA components
    Projecting back using 64 PCA components


# **4. Computing & Assessing a TEP** 

# **(Assignment 4, Deadline 13/11/2026 at 08:00 AM)**

## **4.1. Preparatory Steps**

**Compulsory:** for each preparatory step, students describe its functional significance (that is, why we do it) above the corresponding code cell

Preparatory steps are a set of pre-processing activities that do not fall within any other preprocessing pipeline label. They consist in adjustments done mainly for aesthetic purposes and TEPs interpretability.

In the present preprocessing pipeline these steps are:
1. Low-pass filtering
2. Re-referencing
3. Baseline correction

However, this list is not a dogma and different pipelines might use different preparatory steps.

**Low pass filtering**

This step is done in order to enhance a certain frequency-band of the observed signal. In particular, in this step the filter applied convolves the signal with a kernel that attenuates frequencies above 70 Hz, emphasizing frequencies in the 0-70Hz band (so below 70 Hz). In this case the chosen cut-off value is 70Hz because, excluding high-gamma activity that is usually not of interest, cerebral activity is lower that this value.

In [3]:
post_ica_epochs = mne.read_epochs(fname="data/post_ica_epo.fif", 
                                        preload=True)

low_pass_filtered_data = mne.filter.filter_data(data=post_ica_epochs.get_data(),
                                                sfreq=post_ica_epochs.info["sfreq"],
                                                l_freq=None,
                                                h_freq=70.0,
                                                method="iir",
                                                iir_params=None,
                                                copy=True,
                                                phase="zero")
low_pass_filtered_epochs = mne.EpochsArray(data=low_pass_filtered_data,
                                           info=post_ica_epochs.info,
                                           baseline=None)
low_pass_filtered_tep = low_pass_filtered_epochs.average()
low_pass_filtered_tep.plot();

Reading C:\Users\Giulia\Desktop\brainstim-multimodal\data\post_ica_epo.fif ...
    Found the data of interest:
        t =   -1100.00 ...     500.00 ms
        0 CTF compensation matrices available
Not setting metadata
188 matching events found
No baseline correction applied
0 projection items activated
Setting up low-pass filter at 70 Hz

IIR filter parameters
---------------------
Butterworth lowpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 8 (effective, after forward-backward)
- Cutoff at 70.00 Hz: -6.02 dB

Not setting metadata
188 matching events found
No baseline correction applied
0 projection items activated


**Re-referencing**

EEG data enter the preprocessing pipeline with a built-in reference. This reference is chosen in order to respect practical, low level needs that might not be scientifically optimal. That is because a reference localized in space causes an overestimate of the distant signal and an underestimate of the closer. The goal is then to have a reference that is *less spatially localized as possible* (and that is not an electode of interest of the study) and this is pursued using offline references among which Digitally Linked Mastoids (DLMs) and average reference.

DLMs reference is obtained computing the average of the signal from the right and left mastoids; due to the anatomical location of the mastoids, this average can be considered central both on the medial-lateral axis and on the rostro-caudal axis.

Average reference (that is the one used in this pipeline as shown in the following block of code) is obtained computing the average of the signal from all the electodes; this new reference shows activity as a deviation from the average. 

This step concerns the *representation* of data.

In [4]:
rereferenced_epochs, reference = mne.set_eeg_reference(inst=low_pass_filtered_epochs,
                                                       ref_channels="average") 

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.


**Baseline correction**

The goal of this step is to represent post-stimulus activity as a deviation from baseline activity. This is done first defining a baseline window (start = time of event - 0.100 seconds , end = time of event - 0.002 seconds) and then computing the average of the signal in this time window. This average is considered the baseline activity, the deviation from which is obtained subtracting the pre-stimulus activity from the post-stimulus activity. 

In [5]:
TIME_OF_EVENT = 1.1                             # seconds
baseline_window_start = TIME_OF_EVENT - 0.100   # seconds
baseline_window_end = TIME_OF_EVENT - 0.002     # seconds

baseline_corrected_epochs = rereferenced_epochs.apply_baseline(baseline=(baseline_window_start, baseline_window_end))
baseline_corrected_tep = baseline_corrected_epochs.average()
baseline_corrected_tep.plot();

Applying baseline correction (mode: mean)


## **4.2. Computation of the final TEP** 

The last step of the present preprocessong pipeline consists in the computation of the final TEP. The following block of code crops the signal into epochs for visualization purposes, computes the average of all the TEPs in order to obtain the final TEP, and lastly plots it.

In [6]:
final_epochs = baseline_corrected_epochs.crop(tmin=1.0,
                                              tmax=1.6,
                                              include_tmax=True)
final_tep = final_epochs.average()
final_tep.plot();

# **Closing Remarks**

**Optional:** Student can write here any comment or consideration that goes beyond the minimum required, for example:
- "I think filtering is `{good/bad}` because..."
- "I think these data were particularly `{dirty/clean}` because..."
- "I think that thing X is an issue because..."
- Anything else you might feel like saying about the preprocessing you have done

Do not be afraid to make such comments: there is no right or wrong and they do not count for the vote &mdash; we really just want to provide an outlet for personal opinions. No opinions? No problem!

I think it might be interesting to confront how different choices in the preprocessing pipeline can lead to different final TEPs, for example comparing the TEP obtained with a conservationist approach to the one obtained with a more interventionist one.